# Online Shopping Data Analysis
### Complete End-to-End Project | Python, Pandas, NumPy, Matplotlib, Seaborn, SQL, Power BI

**Objective:** Analyze online shopping data to understand customers, products, purchasing behavior, and peak sales periods.

**Important:** This notebook generates realistic **synthetic sample data**. It is not real customer or company data.

### Project workflow
1. Generate sample customers, products, orders, and order-item data.
2. Inspect and clean data using Pandas and NumPy.
3. Perform exploratory data analysis (EDA).
4. Analyze sales, profit, products, customers, payment methods, geography, and seasonality.
5. Perform RFM customer segmentation and basket analysis.
6. Run SQL queries using SQLite.
7. Export clean tables and analysis results for Power BI.
8. Summarize business insights and recommendations.

**How to run:** Upload this notebook to Google Colab and choose **Runtime → Run all**. The notebook creates an `online_shopping_outputs` folder and downloads can be made from the Colab file panel.

## 1. Import required libraries

In [ ]:
# Import libraries for data manipulation, numerical analysis, visualization, SQL, and file handling.
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

# Create a folder for all project outputs.
OUTPUT_DIR = Path("online_shopping_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Libraries imported successfully.")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

## 2. Generate the synthetic sample dataset

In [ ]:
# Set the size and date range for the synthetic dataset.
NUM_CUSTOMERS = 800
NUM_PRODUCTS = 80
NUM_ORDERS = 6000
START_DATE = pd.Timestamp("2025-01-01")
END_DATE = pd.Timestamp("2026-10-27")

# Create customer records.
cities = ["Delhi", "Mumbai", "Bengaluru", "Hyderabad", "Lucknow",
          "Jaipur", "Pune", "Chennai", "Kolkata", "Ahmedabad"]
states = {
    "Delhi": "Delhi", "Mumbai": "Maharashtra", "Bengaluru": "Karnataka",
    "Hyderabad": "Telangana", "Lucknow": "Uttar Pradesh", "Jaipur": "Rajasthan",
    "Pune": "Maharashtra", "Chennai": "Tamil Nadu", "Kolkata": "West Bengal",
    "Ahmedabad": "Gujarat"
}
customer_ids = [f"CUST{i:05d}" for i in range(1, NUM_CUSTOMERS + 1)]
customer_city = rng.choice(cities, size=NUM_CUSTOMERS)
customers = pd.DataFrame({
    "Customer_ID": customer_ids,
    "Customer_Name": [f"Customer {i:04d}" for i in range(1, NUM_CUSTOMERS + 1)],
    "Age": rng.integers(18, 66, size=NUM_CUSTOMERS),
    "Gender": rng.choice(["Female", "Male", "Prefer not to say"], size=NUM_CUSTOMERS, p=[0.47, 0.48, 0.05]),
    "City": customer_city,
    "State": [states[c] for c in customer_city],
    "Signup_Date": pd.to_datetime(rng.integers(START_DATE.value // 10**9, END_DATE.value // 10**9, size=NUM_CUSTOMERS), unit="s").date
})

# Create product records with categories, prices, and estimated cost.
categories = {
    "Electronics": ["Headphones", "Smartphone", "Keyboard", "Mouse", "Smartwatch", "Speaker", "Power Bank"],
    "Fashion": ["T-Shirt", "Jeans", "Sneakers", "Jacket", "Backpack", "Watch", "Sunglasses"],
    "Home & Kitchen": ["Mixer", "Cookware Set", "Bedsheet", "Lamp", "Storage Box", "Water Bottle", "Toaster"],
    "Beauty": ["Face Wash", "Moisturizer", "Shampoo", "Perfume", "Sunscreen", "Lip Balm", "Hair Dryer"],
    "Books": ["Novel", "Notebook Set", "Exam Guide", "Biography", "Programming Book", "Cookbook", "Journal"]
}
product_rows = []
product_counter = 1
for category, names in categories.items():
    for i in range(NUM_PRODUCTS // len(categories)):
        name = names[i % len(names)]
        base_price = {
            "Electronics": 1800, "Fashion": 900, "Home & Kitchen": 1200,
            "Beauty": 450, "Books": 350
        }[category]
        price = round(float(base_price * rng.uniform(0.35, 2.8)), 2)
        product_rows.append({
            "Product_ID": f"PROD{product_counter:04d}",
            "Product_Name": f"{name} {i+1}",
            "Category": category,
            "Unit_Price": price,
            "Unit_Cost": round(price * float(rng.uniform(0.52, 0.82)), 2)
        })
        product_counter += 1
products = pd.DataFrame(product_rows)

# Generate orders. The probabilities are adjusted to create realistic purchasing patterns.
order_ids = [f"ORD{i:06d}" for i in range(1, NUM_ORDERS + 1)]
order_seconds = rng.integers(START_DATE.value // 10**9, END_DATE.value // 10**9, size=NUM_ORDERS)
order_dates = pd.to_datetime(order_seconds, unit="s").normalize()
order_customer_ids = rng.choice(customer_ids, size=NUM_ORDERS)
payment_methods = rng.choice(["UPI", "Credit Card", "Debit Card", "Cash on Delivery", "Net Banking"], size=NUM_ORDERS,
                             p=[0.38, 0.22, 0.17, 0.15, 0.08])
channels = rng.choice(["Website", "Mobile App", "Marketplace"], size=NUM_ORDERS, p=[0.28, 0.52, 0.20])
order_statuses = rng.choice(["Delivered", "Cancelled", "Returned"], size=NUM_ORDERS, p=[0.91, 0.055, 0.035])

orders = pd.DataFrame({
    "Order_ID": order_ids,
    "Customer_ID": order_customer_ids,
    "Order_Date": order_dates,
    "Payment_Method": payment_methods,
    "Sales_Channel": channels,
    "Order_Status": order_statuses
})

# Create one to four item lines for each order.
item_rows = []
for row in orders.itertuples(index=False):
    number_of_items = int(rng.choice([1, 2, 3, 4], p=[0.50, 0.28, 0.15, 0.07]))
    selected_products = rng.choice(products["Product_ID"].to_numpy(), size=number_of_items, replace=False)
    for product_id in selected_products:
        product = products.loc[products["Product_ID"] == product_id].iloc[0]
        quantity = int(rng.choice([1, 2, 3, 4], p=[0.68, 0.22, 0.08, 0.02]))
        discount_pct = float(rng.choice([0, 0.05, 0.10, 0.15, 0.20], p=[0.35, 0.20, 0.25, 0.15, 0.05]))
        gross_sales = quantity * float(product["Unit_Price"])
        discount_amount = round(gross_sales * discount_pct, 2)
        net_sales = round(gross_sales - discount_amount, 2)
        cost = round(quantity * float(product["Unit_Cost"]), 2)
        item_rows.append({
            "Order_ID": row.Order_ID,
            "Product_ID": product_id,
            "Quantity": quantity,
            "Unit_Price": float(product["Unit_Price"]),
            "Unit_Cost": float(product["Unit_Cost"]),
            "Discount_Pct": discount_pct,
            "Discount_Amount": discount_amount,
            "Sales": net_sales,
            "Cost": cost,
            "Profit": round(net_sales - cost, 2)
        })

order_items = pd.DataFrame(item_rows)

# Add a few intentional data quality issues so that the cleaning section can demonstrate how to handle them.
orders.loc[orders.sample(frac=0.002, random_state=RANDOM_SEED).index, "Payment_Method"] = np.nan
if len(order_items) > 10:
    order_items.loc[order_items.sample(frac=0.001, random_state=RANDOM_SEED).index, "Discount_Pct"] = np.nan

# Save raw source tables.
customers.to_csv(OUTPUT_DIR / "customers_raw.csv", index=False)
products.to_csv(OUTPUT_DIR / "products_raw.csv", index=False)
orders.to_csv(OUTPUT_DIR / "orders_raw.csv", index=False)
order_items.to_csv(OUTPUT_DIR / "order_items_raw.csv", index=False)

print(f"Customers: {len(customers):,}")
print(f"Products: {len(products):,}")
print(f"Orders: {len(orders):,}")
print(f"Order item rows: {len(order_items):,}")

## 3. Preview the data

In [ ]:
# Display a preview of each generated table.
display(customers.head())
display(products.head())
display(orders.head())
display(order_items.head())

# Check the dimensions and data types of the main tables.
print("Customers shape:", customers.shape)
print("Products shape:", products.shape)
print("Orders shape:", orders.shape)
print("Order items shape:", order_items.shape)

## 4. Data quality assessment

In [ ]:
# Inspect missing values, duplicates, and basic numeric summaries before cleaning.
def quality_report(df, table_name):
    report = pd.DataFrame({
        "Column": df.columns,
        "Data_Type": [str(dtype) for dtype in df.dtypes],
        "Missing_Values": [int(df[col].isna().sum()) for col in df.columns],
        "Missing_Percent": [round(df[col].isna().mean() * 100, 2) for col in df.columns]
    })
    print(f"--- {table_name} ---")
    print("Rows:", len(df), "| Duplicate rows:", int(df.duplicated().sum()))
    display(report)

quality_report(customers, "Customers")
quality_report(products, "Products")
quality_report(orders, "Orders")
quality_report(order_items, "Order Items")

## 5. Clean and validate the data

In [ ]:
# Work on copies so that the raw data remains unchanged.
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

# Remove exact duplicate rows and duplicate primary keys.
customers_clean = customers_clean.drop_duplicates(subset=["Customer_ID"]).copy()
products_clean = products_clean.drop_duplicates(subset=["Product_ID"]).copy()
orders_clean = orders_clean.drop_duplicates(subset=["Order_ID"]).copy()
order_items_clean = order_items_clean.drop_duplicates(subset=["Order_ID", "Product_ID"]).copy()

# Convert dates to datetime and numeric columns to numeric types.
customers_clean["Signup_Date"] = pd.to_datetime(customers_clean["Signup_Date"], errors="coerce")
orders_clean["Order_Date"] = pd.to_datetime(orders_clean["Order_Date"], errors="coerce")
for col in ["Age"]:
    customers_clean[col] = pd.to_numeric(customers_clean[col], errors="coerce")
for col in ["Unit_Price", "Unit_Cost"]:
    products_clean[col] = pd.to_numeric(products_clean[col], errors="coerce")
numeric_item_cols = ["Quantity", "Unit_Price", "Unit_Cost", "Discount_Pct", "Discount_Amount", "Sales", "Cost", "Profit"]
for col in numeric_item_cols:
    order_items_clean[col] = pd.to_numeric(order_items_clean[col], errors="coerce")

# Fill missing categorical values with the most frequent value.
orders_clean["Payment_Method"] = orders_clean["Payment_Method"].fillna(orders_clean["Payment_Method"].mode().iloc[0])

# Fill missing discount percentage with zero and recompute dependent financial fields.
order_items_clean["Discount_Pct"] = order_items_clean["Discount_Pct"].fillna(0).clip(0, 0.80)
order_items_clean["Quantity"] = order_items_clean["Quantity"].fillna(1).clip(lower=1).astype(int)

# Recalculate sales, discount, cost, and profit to ensure financial consistency.
order_items_clean["Discount_Amount"] = (
    order_items_clean["Quantity"] * order_items_clean["Unit_Price"] * order_items_clean["Discount_Pct"]
).round(2)
order_items_clean["Sales"] = (
    order_items_clean["Quantity"] * order_items_clean["Unit_Price"] - order_items_clean["Discount_Amount"]
).round(2)
order_items_clean["Cost"] = (order_items_clean["Quantity"] * order_items_clean["Unit_Cost"]).round(2)
order_items_clean["Profit"] = (order_items_clean["Sales"] - order_items_clean["Cost"]).round(2)

# Remove rows with invalid keys or invalid financial values.
order_items_clean = order_items_clean[
    order_items_clean["Order_ID"].notna()
    & order_items_clean["Product_ID"].notna()
    & order_items_clean["Sales"].notna()
    & order_items_clean["Sales"].ge(0)
].copy()

# Keep only rows that reference existing customers, products, and orders.
orders_clean = orders_clean[orders_clean["Customer_ID"].isin(customers_clean["Customer_ID"])].copy()
order_items_clean = order_items_clean[
    order_items_clean["Order_ID"].isin(orders_clean["Order_ID"])
    & order_items_clean["Product_ID"].isin(products_clean["Product_ID"])
].copy()

# Join dimensions onto order-item data for analysis.
analysis_df = (
    order_items_clean
    .merge(orders_clean, on="Order_ID", how="left", validate="many_to_one")
    .merge(products_clean[["Product_ID", "Product_Name", "Category"]], on="Product_ID", how="left", validate="many_to_one")
    .merge(customers_clean[["Customer_ID", "Age", "Gender", "City", "State"]], on="Customer_ID", how="left", validate="many_to_one")
)

# Exclude cancelled orders from recognized sales and profit analysis.
analysis_df["Is_Completed_Order"] = analysis_df["Order_Status"].eq("Delivered")
analysis_df["Recognized_Sales"] = np.where(analysis_df["Is_Completed_Order"], analysis_df["Sales"], 0)
analysis_df["Recognized_Profit"] = np.where(analysis_df["Is_Completed_Order"], analysis_df["Profit"], 0)

print("Cleaned table shapes:")
print("Customers:", customers_clean.shape)
print("Products:", products_clean.shape)
print("Orders:", orders_clean.shape)
print("Order items:", order_items_clean.shape)
print("Joined analysis data:", analysis_df.shape)
display(analysis_df.head())

## 6. Create useful analytical features

In [ ]:
# Add date-related fields and order-level metrics for trend analysis.
analysis_df["Year"] = analysis_df["Order_Date"].dt.year
analysis_df["Month"] = analysis_df["Order_Date"].dt.month
analysis_df["Month_Name"] = analysis_df["Order_Date"].dt.strftime("%b")
analysis_df["Year_Month"] = analysis_df["Order_Date"].dt.to_period("M").astype(str)
analysis_df["Day_Name"] = analysis_df["Order_Date"].dt.day_name()
analysis_df["Quarter"] = "Q" + analysis_df["Order_Date"].dt.quarter.astype(str)

# Build a one-row-per-order table from item-level data.
order_summary = (
    analysis_df.groupby(["Order_ID", "Customer_ID", "Order_Date", "Payment_Method", "Sales_Channel", "Order_Status"], as_index=False)
    .agg(Order_Sales=("Sales", "sum"), Order_Profit=("Profit", "sum"),
         Total_Quantity=("Quantity", "sum"), Distinct_Products=("Product_ID", "nunique"))
)
order_summary["Recognized_Sales"] = np.where(order_summary["Order_Status"].eq("Delivered"), order_summary["Order_Sales"], 0)
order_summary["Recognized_Profit"] = np.where(order_summary["Order_Status"].eq("Delivered"), order_summary["Order_Profit"], 0)

# Save cleaned and joined data for reuse.
customers_clean.to_csv(OUTPUT_DIR / "customers_clean.csv", index=False)
products_clean.to_csv(OUTPUT_DIR / "products_clean.csv", index=False)
orders_clean.to_csv(OUTPUT_DIR / "orders_clean.csv", index=False)
order_items_clean.to_csv(OUTPUT_DIR / "order_items_clean.csv", index=False)
analysis_df.to_csv(OUTPUT_DIR / "analysis_data_powerbi.csv", index=False)
order_summary.to_csv(OUTPUT_DIR / "order_summary_powerbi.csv", index=False)

display(order_summary.head())

## 7. Overall business KPIs

In [ ]:
# Calculate top-level metrics using delivered orders as recognized business activity.
delivered = order_summary[order_summary["Order_Status"].eq("Delivered")].copy()
total_sales = delivered["Order_Sales"].sum()
total_profit = delivered["Order_Profit"].sum()
total_orders = delivered["Order_ID"].nunique()
unique_customers = delivered["Customer_ID"].nunique()
average_order_value = total_sales / total_orders if total_orders else 0
profit_margin = (total_profit / total_sales * 100) if total_sales else 0
total_units = delivered["Total_Quantity"].sum()

kpis = pd.DataFrame({
    "Metric": ["Recognized Sales", "Recognized Profit", "Delivered Orders", "Unique Customers",
               "Average Order Value", "Profit Margin (%)", "Units Sold"],
    "Value": [total_sales, total_profit, total_orders, unique_customers,
              average_order_value, profit_margin, total_units]
})
display(kpis.style.format({"Value": "{:,.2f}"}))

## 8. Sales trends and peak periods

In [ ]:
# Aggregate delivered sales and profit by month.
monthly = (
    delivered.assign(Year_Month=delivered["Order_Date"].dt.to_period("M").astype(str))
    .groupby("Year_Month", as_index=False)
    .agg(Sales=("Order_Sales", "sum"), Profit=("Order_Profit", "sum"), Orders=("Order_ID", "nunique"))
)

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly, x="Year_Month", y="Sales", marker="o")
plt.title("Monthly Recognized Sales")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly, x="Year_Month", y="Orders", marker="o")
plt.title("Monthly Delivered Orders")
plt.xlabel("Month")
plt.ylabel("Number of Orders")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

# Compare weekday sales to identify stronger shopping days.
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_sales = analysis_df[analysis_df["Order_Status"].eq("Delivered")].groupby("Day_Name")["Sales"].sum().reindex(weekday_order)

plt.figure(figsize=(10, 5))
sns.barplot(x=weekday_sales.index, y=weekday_sales.values)
plt.title("Recognized Sales by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Sales")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

display(monthly.sort_values("Sales", ascending=False).head(10))

## 9. Product and category performance

In [ ]:
# Rank product categories by sales, profit, and quantity.
category_performance = (
    analysis_df[analysis_df["Order_Status"].eq("Delivered")]
    .groupby("Category", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Units_Sold=("Quantity", "sum"),
         Orders=("Order_ID", "nunique"))
    .sort_values("Sales", ascending=False)
)
category_performance["Profit_Margin_Pct"] = np.where(
    category_performance["Sales"] > 0,
    category_performance["Profit"] / category_performance["Sales"] * 100, 0
)
display(category_performance)

plt.figure(figsize=(10, 5))
sns.barplot(data=category_performance, x="Category", y="Sales")
plt.title("Sales by Product Category")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

product_performance = (
    analysis_df[analysis_df["Order_Status"].eq("Delivered")]
    .groupby(["Product_ID", "Product_Name", "Category"], as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Units_Sold=("Quantity", "sum"))
    .sort_values("Sales", ascending=False)
)
display(product_performance.head(15))

plt.figure(figsize=(12, 6))
top_products = product_performance.head(10).sort_values("Sales")
sns.barplot(data=top_products, x="Sales", y="Product_Name")
plt.title("Top 10 Products by Sales")
plt.tight_layout()
plt.show()

## 10. Customer behavior and geographic analysis

In [ ]:
# Summarize customer activity for delivered orders.
customer_summary = (
    delivered.groupby("Customer_ID", as_index=False)
    .agg(Orders=("Order_ID", "nunique"), Sales=("Order_Sales", "sum"),
         Profit=("Order_Profit", "sum"), Units=("Total_Quantity", "sum"),
         Last_Order_Date=("Order_Date", "max"))
    .merge(customers_clean[["Customer_ID", "Age", "Gender", "City", "State"]], on="Customer_ID", how="left")
)
customer_summary["Average_Order_Value"] = customer_summary["Sales"] / customer_summary["Orders"]
display(customer_summary.sort_values("Sales", ascending=False).head(15))

# Analyze sales by city.
city_performance = (
    analysis_df[analysis_df["Order_Status"].eq("Delivered")]
    .groupby(["City", "State"], as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order_ID", "nunique"))
    .sort_values("Sales", ascending=False)
)
display(city_performance)

plt.figure(figsize=(10, 5))
sns.barplot(data=city_performance.head(10), x="Sales", y="City")
plt.title("Top Cities by Recognized Sales")
plt.tight_layout()
plt.show()

# Compare customer metrics by age group.
customer_summary["Age_Group"] = pd.cut(
    customer_summary["Age"], bins=[17, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"]
)
age_performance = customer_summary.groupby("Age_Group", observed=False, as_index=False).agg(
    Customers=("Customer_ID", "nunique"), Sales=("Sales", "sum"), Average_Customer_Sales=("Sales", "mean")
)
display(age_performance)

## 11. Payment method, channel, discount, and order status analysis

In [ ]:
# Compare delivered sales by payment method and sales channel.
payment_performance = (
    analysis_df[analysis_df["Order_Status"].eq("Delivered")]
    .groupby("Payment_Method", as_index=False)
    .agg(Sales=("Sales", "sum"), Orders=("Order_ID", "nunique"))
    .sort_values("Sales", ascending=False)
)
channel_performance = (
    analysis_df[analysis_df["Order_Status"].eq("Delivered")]
    .groupby("Sales_Channel", as_index=False)
    .agg(Sales=("Sales", "sum"), Orders=("Order_ID", "nunique"))
    .sort_values("Sales", ascending=False)
)
status_performance = orders_clean["Order_Status"].value_counts().rename_axis("Order_Status").reset_index(name="Orders")

display(payment_performance)
display(channel_performance)
display(status_performance)

plt.figure(figsize=(9, 5))
sns.barplot(data=payment_performance, x="Payment_Method", y="Sales")
plt.title("Sales by Payment Method")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=channel_performance, x="Sales_Channel", y="Sales")
plt.title("Sales by Sales Channel")
plt.tight_layout()
plt.show()

# Compare average order-item sales across discount levels.
discount_analysis = (
    analysis_df[analysis_df["Order_Status"].eq("Delivered")]
    .assign(Discount_Band=lambda d: pd.cut(
        d["Discount_Pct"], bins=[-0.001, 0, 0.05, 0.10, 0.15, 0.20, 1],
        labels=["0%", "1-5%", "6-10%", "11-15%", "16-20%", ">20%"]
    ))
    .groupby("Discount_Band", observed=False, as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Item_Lines=("Order_ID", "count"))
)
display(discount_analysis)

## 12. Customer RFM analysis and segmentation

In [ ]:
# Calculate Recency, Frequency, and Monetary value from delivered orders.
analysis_date = orders_clean["Order_Date"].max() + pd.Timedelta(days=1)
rfm = (
    delivered.groupby("Customer_ID", as_index=False)
    .agg(Last_Purchase=("Order_Date", "max"),
         Frequency=("Order_ID", "nunique"),
         Monetary=("Order_Sales", "sum"))
)
rfm["Recency"] = (analysis_date - rfm["Last_Purchase"]).dt.days

# Score each RFM metric from 1 to 5. Lower recency is better, so reverse its rank.
rfm["R_Score"] = pd.qcut(rfm["Recency"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM_Score"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)

# Apply simple, explainable customer segment rules.
def assign_segment(row):
    if row["R_Score"] >= 4 and row["F_Score"] >= 4 and row["M_Score"] >= 4:
        return "Champions"
    if row["R_Score"] >= 3 and row["F_Score"] >= 3:
        return "Loyal Customers"
    if row["R_Score"] >= 4 and row["F_Score"] <= 2:
        return "Recent Customers"
    if row["R_Score"] <= 2 and row["F_Score"] >= 3:
        return "At Risk"
    if row["F_Score"] == 1:
        return "Need Attention"
    return "Potential Loyalists"

rfm["Customer_Segment"] = rfm.apply(assign_segment, axis=1)
rfm = rfm.merge(customers_clean[["Customer_ID", "City", "State", "Age", "Gender"]], on="Customer_ID", how="left")
segment_summary = rfm.groupby("Customer_Segment", as_index=False).agg(
    Customers=("Customer_ID", "nunique"), Sales=("Monetary", "sum"),
    Average_Recency=("Recency", "mean"), Average_Frequency=("Frequency", "mean")
).sort_values("Sales", ascending=False)

display(segment_summary)
display(rfm.sort_values("Monetary", ascending=False).head(10))

plt.figure(figsize=(10, 5))
sns.barplot(data=segment_summary, x="Customer_Segment", y="Customers")
plt.title("Customers by RFM Segment")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 13. Basket analysis: frequently purchased product pairs

In [ ]:
# Identify product pairs that appear together in the same order.
# This is a simple co-occurrence analysis, not a full association-rule model.
delivered_items = analysis_df[analysis_df["Order_Status"].eq("Delivered")][["Order_ID", "Product_Name"]].drop_duplicates()
basket_sets = delivered_items.groupby("Order_ID")["Product_Name"].apply(lambda x: sorted(set(x))).tolist()

from itertools import combinations
pair_counts = {}
for basket in basket_sets:
    for pair in combinations(basket, 2):
        pair_counts[pair] = pair_counts.get(pair, 0) + 1

basket_pairs = pd.DataFrame(
    [{"Product_A": pair[0], "Product_B": pair[1], "Orders_Together": count}
     for pair, count in pair_counts.items()]
)
if not basket_pairs.empty:
    basket_pairs = basket_pairs.sort_values("Orders_Together", ascending=False).reset_index(drop=True)
display(basket_pairs.head(20))
basket_pairs.to_csv(OUTPUT_DIR / "basket_product_pairs.csv", index=False)

## 14. Correlation and distribution checks

In [ ]:
# Explore relationships among item-level numeric measures.
numeric_cols = ["Quantity", "Unit_Price", "Discount_Pct", "Discount_Amount", "Sales", "Cost", "Profit"]
correlation_matrix = analysis_df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(9, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Numeric Variables")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.histplot(data=delivered, x="Order_Sales", bins=40, kde=True)
plt.title("Distribution of Order Values")
plt.xlabel("Order Sales")
plt.tight_layout()
plt.show()

## 15. SQL analysis using SQLite

In [ ]:
# Load the cleaned tables into a local SQLite database.
db_path = OUTPUT_DIR / "online_shopping.db"
with sqlite3.connect(db_path) as conn:
    customers_clean.to_sql("customers", conn, if_exists="replace", index=False)
    products_clean.to_sql("products", conn, if_exists="replace", index=False)
    orders_clean.to_sql("orders", conn, if_exists="replace", index=False)
    order_items_clean.to_sql("order_items", conn, if_exists="replace", index=False)

# Helper function to run a SQL query and display the result.
def run_sql(query):
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(query, conn)

# Query 1: Monthly sales and profit for delivered orders.
sql_monthly_sales = """
SELECT strftime('%Y-%m', o.Order_Date) AS Year_Month,
       ROUND(SUM(oi.Sales), 2) AS Sales,
       ROUND(SUM(oi.Profit), 2) AS Profit,
       COUNT(DISTINCT o.Order_ID) AS Delivered_Orders
FROM orders o
JOIN order_items oi ON o.Order_ID = oi.Order_ID
WHERE o.Order_Status = 'Delivered'
GROUP BY strftime('%Y-%m', o.Order_Date)
ORDER BY Year_Month;
"""
display(run_sql(sql_monthly_sales))

# Query 2: Top products by sales.
sql_top_products = """
SELECT p.Product_Name, p.Category,
       ROUND(SUM(oi.Sales), 2) AS Sales,
       SUM(oi.Quantity) AS Units_Sold
FROM order_items oi
JOIN products p ON oi.Product_ID = p.Product_ID
JOIN orders o ON oi.Order_ID = o.Order_ID
WHERE o.Order_Status = 'Delivered'
GROUP BY p.Product_ID, p.Product_Name, p.Category
ORDER BY Sales DESC
LIMIT 10;
"""
display(run_sql(sql_top_products))

# Query 3: Customer lifetime value proxy from delivered orders.
sql_top_customers = """
SELECT c.Customer_ID, c.City,
       COUNT(DISTINCT o.Order_ID) AS Orders,
       ROUND(SUM(oi.Sales), 2) AS Total_Sales
FROM customers c
JOIN orders o ON c.Customer_ID = o.Customer_ID
JOIN order_items oi ON o.Order_ID = oi.Order_ID
WHERE o.Order_Status = 'Delivered'
GROUP BY c.Customer_ID, c.City
ORDER BY Total_Sales DESC
LIMIT 10;
"""
display(run_sql(sql_top_customers))

# Query 4: Payment method performance.
sql_payment = """
SELECT o.Payment_Method,
       COUNT(DISTINCT o.Order_ID) AS Orders,
       ROUND(SUM(oi.Sales), 2) AS Sales
FROM orders o
JOIN order_items oi ON o.Order_ID = oi.Order_ID
WHERE o.Order_Status = 'Delivered'
GROUP BY o.Payment_Method
ORDER BY Sales DESC;
"""
display(run_sql(sql_payment))

# Query 5: Order status counts.
sql_status = """
SELECT Order_Status, COUNT(*) AS Order_Count
FROM orders
GROUP BY Order_Status
ORDER BY Order_Count DESC;
"""
display(run_sql(sql_status))

# Save the main SQL queries to a text file so they can be submitted separately.
sql_text = """-- Online Shopping Data Analysis: SQL Queries
-- Database tables: customers, products, orders, order_items

""" + "\n\n".join([
    "-- Query 1: Monthly sales and profit\n" + sql_monthly_sales,
    "-- Query 2: Top products by sales\n" + sql_top_products,
    "-- Query 3: Top customers by delivered sales\n" + sql_top_customers,
    "-- Query 4: Sales by payment method\n" + sql_payment,
    "-- Query 5: Order status counts\n" + sql_status
])
(OUTPUT_DIR / "online_shopping_queries.sql").write_text(sql_text, encoding="utf-8")
print("SQLite database and SQL query file created.")

## 16. Prepare Power BI export tables

In [ ]:
# Export clean, analysis-ready tables for Power BI.
# Recommended model: customers (1) -> orders (*) -> order_items (*)
#                    products (1) -> order_items (*)
powerbi_exports = {
    "PowerBI_Customers.csv": customers_clean,
    "PowerBI_Products.csv": products_clean,
    "PowerBI_Orders.csv": orders_clean,
    "PowerBI_OrderItems.csv": order_items_clean,
    "PowerBI_AnalysisData.csv": analysis_df,
    "PowerBI_OrderSummary.csv": order_summary,
    "PowerBI_RFM_Segments.csv": rfm,
    "PowerBI_CategoryPerformance.csv": category_performance,
    "PowerBI_MonthlySales.csv": monthly,
    "PowerBI_CityPerformance.csv": city_performance,
    "PowerBI_PaymentPerformance.csv": payment_performance,
    "PowerBI_ChannelPerformance.csv": channel_performance,
    "PowerBI_BasketPairs.csv": basket_pairs
}
for filename, frame in powerbi_exports.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)

print("Power BI files exported:")
for filename in powerbi_exports:
    print(" -", filename)

## 17. Power BI dashboard build guide

### Recommended data model
Import the CSV files from the `online_shopping_outputs` folder into Power BI Desktop.

Create these relationships:
- `customers[Customer_ID]` **1 → many** `orders[Customer_ID]`
- `orders[Order_ID]` **1 → many** `order_items[Order_ID]`
- `products[Product_ID]` **1 → many** `order_items[Product_ID]`

Use the dimension tables for slicers and the order-item fact table for product-level analysis. Avoid creating multiple active paths between the same tables.

### Suggested report pages
1. **Executive Overview:** Total Sales, Total Profit, Delivered Orders, Unique Customers, Average Order Value; monthly sales line chart; sales by category.
2. **Product Analysis:** Sales and profit by category; top 10 products; units sold; discount analysis.
3. **Customer Analysis:** RFM segments; sales by city/state; customer age group; top customers.
4. **Purchasing Behavior:** Payment method, sales channel, order status, weekday trends, basket product pairs.

### Suggested DAX measures
Create measures in Power BI after importing the data. These examples assume the table names are `orders` and `order_items`.

```DAX
Total Sales =
CALCULATE(
    SUM(order_items[Sales]),
    orders[Order_Status] = "Delivered"
)

Total Profit =
CALCULATE(
    SUM(order_items[Profit]),
    orders[Order_Status] = "Delivered"
)

Delivered Orders =
CALCULATE(
    DISTINCTCOUNT(orders[Order_ID]),
    orders[Order_Status] = "Delivered"
)

Average Order Value =
DIVIDE([Total Sales], [Delivered Orders], 0)

Profit Margin % =
DIVIDE([Total Profit], [Total Sales], 0) * 100

Unique Customers =
CALCULATE(
    DISTINCTCOUNT(orders[Customer_ID]),
    orders[Order_Status] = "Delivered"
)
```

**Note:** The notebook prepares the Power BI datasets and build instructions. The `.pbix` report itself must be created in Power BI Desktop; Colab cannot generate a finished native Power BI report file directly.

## 18. Automated business insights

In [ ]:
# Generate concise findings from the actual synthetic data so the conclusion is data-driven.
top_category = category_performance.sort_values("Sales", ascending=False).iloc[0]
top_product = product_performance.sort_values("Sales", ascending=False).iloc[0]
top_city = city_performance.sort_values("Sales", ascending=False).iloc[0]
top_payment = payment_performance.sort_values("Sales", ascending=False).iloc[0]
top_month = monthly.sort_values("Sales", ascending=False).iloc[0]
top_segment = segment_summary.sort_values("Sales", ascending=False).iloc[0]

insights = [
    f"Recognized sales from delivered orders: {total_sales:,.2f}.",
    f"Recognized profit from delivered orders: {total_profit:,.2f}, with a profit margin of {profit_margin:.2f}%.",
    f"Delivered orders: {total_orders:,}; unique purchasing customers: {unique_customers:,}.",
    f"Average order value: {average_order_value:,.2f}.",
    f"Top category by sales: {top_category['Category']} ({top_category['Sales']:,.2f}).",
    f"Top product by sales: {top_product['Product_Name']} ({top_product['Sales']:,.2f}).",
    f"Top city by sales: {top_city['City']} ({top_city['Sales']:,.2f}).",
    f"Top payment method by sales: {top_payment['Payment_Method']} ({top_payment['Sales']:,.2f}).",
    f"Highest-sales month in the sample: {top_month['Year_Month']} ({top_month['Sales']:,.2f}).",
    f"Highest-sales RFM segment: {top_segment['Customer_Segment']} ({top_segment['Sales']:,.2f})."
]
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

print("\nBusiness recommendations:")
print("1. Maintain inventory for high-performing categories and products.")
print("2. Test targeted offers for customers in the At Risk and Need Attention segments.")
print("3. Compare discounts with profit margin before increasing promotions.")
print("4. Review peak months and plan inventory, staffing, and marketing in advance.")
print("5. Compare channel and payment performance to improve the customer checkout experience.")

## 19. Export a concise KPI and findings report

In [ ]:
# Save the main KPIs and generated insights for the written report.
kpis.to_csv(OUTPUT_DIR / "project_kpis.csv", index=False)
(OUTPUT_DIR / "business_insights.txt").write_text(
    "ONLINE SHOPPING DATA ANALYSIS - BUSINESS INSIGHTS\n\n"
    + "\n".join(f"{i}. {item}" for i, item in enumerate(insights, 1))
    + "\n\nBusiness recommendations:\n"
    + "1. Maintain inventory for high-performing categories and products.\n"
    + "2. Test targeted offers for customers in the At Risk and Need Attention segments.\n"
    + "3. Compare discounts with profit margin before increasing promotions.\n"
    + "4. Review peak months and plan inventory, staffing, and marketing in advance.\n"
    + "5. Compare channel and payment performance to improve the customer checkout experience.\n",
    encoding="utf-8"
)

# Create a single ZIP archive for convenient download from Colab.
import shutil
zip_path = shutil.make_archive("online_shopping_outputs", "zip", OUTPUT_DIR)

print("Project output files are ready.")
print("Output directory:", OUTPUT_DIR.resolve())
print("ZIP archive:", Path(zip_path).resolve())
print("Files created:", len(list(OUTPUT_DIR.iterdir())))

## 20. Final project conclusion

This project demonstrates an end-to-end online shopping analytics workflow using Python, Pandas, NumPy, Matplotlib, Seaborn, SQL, and Power BI-ready exports. It covers data generation, quality checks, cleaning, exploratory analysis, sales and profit trends, customer segmentation, basket co-occurrence, SQL querying, and business recommendations.

**Limitations:** All generated records are synthetic and designed for educational demonstration. Results should not be interpreted as real market findings. The RFM labels are rule-based, and basket analysis counts product pairs rather than applying a full association-rule algorithm.

**Submission checklist**
- [ ] Run all notebook cells successfully in Google Colab.
- [ ] Download the `online_shopping_outputs.zip` file.
- [ ] Import the Power BI CSV files into Power BI Desktop.
- [ ] Build the report pages and save the `.pbix` file.
- [ ] Add screenshots of key charts/dashboard to the final report.
- [ ] Be ready to explain data cleaning, KPIs, SQL joins, RFM analysis, and business insights in viva.